<a href="https://colab.research.google.com/github/ilistopad303/movie-predictor-BANA7075/blob/Jayweil32-patch-1/Movie_ROI_Prediction_Model_Version_2_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# load in libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import warnings
import joblib

warnings.filterwarnings("ignore")
cleaned_data_file = "../data/processed/movie_dataset_cleaned.parquet"

In [ ]:
# load in dataset
# movie_dataset = pd.read_csv('movie_dataset_cleaned.csv')
movie_dataset = pd.read_parquet(cleaned_data_file)

In [ ]:
# show column names
print("\nCOLUMNS:")
print(movie_dataset.columns.tolist())

In [ ]:
# summary stats for numeric columns
print("\nDESCRIBE:")
display(movie_dataset.describe())

In [ ]:
# check missing values count
print("\nMISSING VALUES:")
print(movie_dataset.isnull().sum())

In [ ]:
# check duplicate rows
print("\nDUPLICATE ROWS:", movie_dataset.duplicated().sum())

In [ ]:
# copy dataset and create target
df = movie_dataset.copy()

In [ ]:
# keep only rows with positive budget
df = df[df['budget'] > 0]

# create ROI target
df['ROI'] = (df['revenue'] - df['budget']) / df['budget']

# -> log (budget was dominating before)
df['log_budget'] = np.log(df['budget'])

In [ ]:
# create release quarter
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['release_quarter'] = df['release_date'].dt.quarter.astype(str)

In [ ]:
# define core genre features
core_genres = [
    'Drama',
    'Comedy',
    'Romance',
    'Horror',
    'Thriller',
    'Crime',
    'Action',
    'Adventure',
    'Science Fiction',
    'Fantasy',
    'Animation',
    'Documentary'
]

In [ ]:
# create binary genre columns
for genre in core_genres:
    df[genre] = df['genres'].apply(lambda x: 1 if genre in x else 0)

In [ ]:
# fill in missing director values
df['director'] = df['director'].fillna('Unknown')

# frequency encode director
df['director_count'] = df['director'].map(df['director'].value_counts())

In [ ]:
# select features and target
features = [
    'log_budget',
    'runtime',
    'release_quarter',
    'original_language',
    'director_count'
] + core_genres

target = 'ROI'

X = df[features]
y = df[target]

In [ ]:
# train/split test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

In [ ]:
# save processed train/test data
train_processed = X_train.copy()
train_processed[target] = y_train

test_processed = X_test.copy()
test_processed[target] = y_test

train_processed.to_csv("train_processed_v2.csv", index=False)
test_processed.to_csv("test_processed_v2.csv", index=False)

print("Processed train/test files saved.")

In [ ]:
# preprocessing pipeline
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

categorical_features = ['release_quarter', 'original_language']
numeric_features = ['log_budget', 'runtime', 'director_count'] + core_genres

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

In [ ]:
# train random forest model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
def rf_model():
    from sklearn.ensemble import RandomForestRegressor
    
    rf_model = Pipeline([
        ('preprocessor', preprocessor),
        ('model', RandomForestRegressor(
            n_estimators=200,
            max_depth=10,
            random_state=123
        ))
    ])
    
    rf_model.fit(X_train, y_train)
    
    y_pred = rf_model.predict(X_test)
    return y_pred
# evaluation metrics
y_pred = rf_model()
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Random Forest (Pre-Release Model (v2)")
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2:", r2_score(y_test, y_pred))

In [ ]:
# save model version
joblib.dump(rf_model, "random_forest_roi_v2.pkl")
print("Model saved as random_forest_roi_v2.pkl")

In [ ]:
# experiment tracking
experiment_log = pd.DataFrame([{
    "model_name": "RandomForestRegression",
    "version": "v2",
    "n_estimators": 200,
    "max_depth": 10,
    "test_size": 0.20,
    "random_state": 123,
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2
}])

experiment_log.to_csv("experiment_log_v2.csv", index=False)
display(experiment_log)

In [ ]:
# feature importance
feature_names = (
    numeric_features +
    list(rf_model.named_steps['preprocessor']
         .named_transformers_['cat']
         .named_steps['onehot']
         .get_feature_names_out(categorical_features))
)

importances = rf_model.named_steps['model'].feature_importances_

feat_imp = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print(feat_imp.head(10))

In [ ]:
# LASSO model for comparison
from sklearn.linear_model import LassoCV

lasso_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LassoCV(
        alphas=np.logspace(-4, 1, 100),
        cv=5,
        random_state=123,
        max_iter=10000
    ))
])

lasso_model.fit(X_train, y_train)

y_pred_lasso = lasso_model.predict(X_test)

lasso_mae = mean_absolute_error(y_test, y_pred_lasso)
lasso_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
lasso_r2 = r2_score(y_test, y_pred_lasso)

print("\nLASSO ROI Model")
print("Best Alpha:", lasso_model.named_steps["model"].alpha_)
print("MAE:", lasso_mae)
print("RMSE:", lasso_rmse)
print("R2:", lasso_r2)

In [ ]:
# LASSO feature importance

lasso_feature_names = (
    numeric_features +
    list(
        lasso_model.named_steps["preprocessor"]
        .named_transformers_["cat"]
        .named_steps["onehot"]
        .get_feature_names_out(categorical_features)
    )
)

lasso_coefficients = lasso_model.named_steps["model"].coef_

lasso_coef_df = pd.DataFrame({
    "feature": lasso_feature_names,
    "coefficient": lasso_coefficients
}).sort_values(by="coefficient", key=abs, ascending=False)

display(lasso_coef_df.head(10))

In [ ]:
# random forest model and LASSO model comparsion
comparison = pd.DataFrame({
    "Model": ["Random Forest", "LASSO"],
    "MAE": [mae, lasso_mae],
    "RMSE": [rmse, lasso_rmse],
    "R2": [r2, lasso_r2]
})

display(comparison)

The Random Forest model outperformed LASSO across all evaluation metrics, achieving lower mean absolute error (MAE = 4.51 vs. 6.92), lower root mean squared error (RMSE = 16.76 vs. 19.04), and a higher R^2 value (0.271 vs. 0.060).

In [ ]:
# calculator v2

budget_input = 100000000
runtime_input = 120
release_quarter_input = "3"
language_input = "en"
director_count_input = 5

new_movie = pd.DataFrame({
    "log_budget": [np.log(budget_input)],
    "runtime": [runtime_input],
    "release_quarter": [release_quarter_input],
    "original_language": [language_input],
    "director_count": [director_count_input],

    "Drama": [0],
    "Comedy": [0],
    "Romance": [0],
    "Horror": [0],
    "Thriller": [1],
    "Crime": [0],
    "Action": [1],
    "Adventure": [1],
    "Science Fiction": [1],
    "Fantasy": [0],
    "Animation": [0],
    "Documentary": [0]
})

# Point prediction
pred_roi = rf_model.predict(new_movie)[0]
pred_revenue = budget_input * (1 + pred_roi)
pred_profit = pred_revenue - budget_input

# Build residual-based range
# This uses your test-set prediction errors to create a more realistic interval
test_pred = rf_model.predict(X_test)
residuals = y_test - test_pred

lower_error = np.percentile(residuals, 25)
upper_error = np.percentile(residuals, 75)

lower_roi = max(-1, pred_roi + lower_error)
upper_roi = pred_roi + upper_error

lower_revenue = max(0, budget_input * (1 + lower_roi))
upper_revenue = max(0, budget_input * (1 + upper_roi))

print("MOVIE ROI CALCULATOR")
print("-" * 40)
print(f"Budget: ${budget_input:,.0f}")
print(f"Runtime: {runtime_input} minutes")
print(f"Release Quarter: Q{release_quarter_input}")
print(f"Original Language: {language_input}")
print(f"Director Count: {director_count_input}")
print("-" * 40)
print(f"Predicted ROI: {pred_roi:.2f} ({pred_roi * 100:.1f}%)")
print(f"Estimated Revenue: ${pred_revenue:,.0f}")
print(f"Estimated Profit: ${pred_profit:,.0f}")
print(f"Revenue Range: ${lower_revenue:,.0f} - ${upper_revenue:,.0f}")

if pred_roi > 0:
    print("Prediction: PROFITABLE")
else:
    print("Prediction: NOT PROFITABLE")

In [ ]:
# Front-end interfacing

import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np

# Error bounds from your model
lower_error = -1.5
upper_error = 1.8

# Input widgets
budget_widget = widgets.IntText(value=100000000, description="Budget:")
runtime_widget = widgets.IntText(value=120, description="Runtime:")
quarter_widget = widgets.Dropdown(options=["1", "2", "3", "4"], value="3", description="Quarter:")
language_widget = widgets.Text(value="en", description="Language:")
director_widget = widgets.IntText(value=5, description="Director Ct:")

genre_options = [
    "Drama", "Comedy", "Romance", "Horror", "Thriller", "Crime",
    "Action", "Adventure", "Science Fiction", "Fantasy",
    "Animation", "Documentary"
]

genre_widget = widgets.SelectMultiple(
    options=genre_options,
    value=("Thriller", "Action", "Adventure", "Science Fiction"),
    description="Genres:",
    rows=8
)

predict_button = widgets.Button(description="Predict ROI", button_style="success")
output = widgets.Output()

def predict_movie(_):
    with output:
        clear_output()

        budget_input = budget_widget.value
        runtime_input = runtime_widget.value
        release_quarter_input = quarter_widget.value
        language_input = language_widget.value.strip().lower()
        director_count_input = director_widget.value
        selected_genres = list(genre_widget.value)

        errors = []

        if budget_input <= 0:
            errors.append("Budget must be greater than 0.")
        if runtime_input <= 0:
            errors.append("Runtime must be greater than 0.")
        if director_count_input < 0:
            errors.append("Director count must be 0 or greater.")
        if language_input == "":
            errors.append("Language cannot be blank.")
        if len(selected_genres) == 0:
            errors.append("Select at least one genre.")

        if errors:
            for err in errors:
                print(err)
            return

        genre_data = {genre: [1 if genre in selected_genres else 0] for genre in genre_options}

        new_movie = pd.DataFrame({
            "log_budget": [np.log(budget_input)],
            "runtime": [runtime_input],
            "release_quarter": [release_quarter_input],
            "original_language": [language_input],
            "director_count": [director_count_input],
            **genre_data
        })

        pred_roi = rf_model.predict(new_movie)[0]
        pred_revenue = budget_input * (1 + pred_roi)
        pred_profit = pred_revenue - budget_input

        lower_roi = max(-1, pred_roi + lower_error)
        upper_roi = pred_roi + upper_error

        lower_revenue = max(0, budget_input * (1 + lower_roi))
        upper_revenue = max(0, budget_input * (1 + upper_roi))

        print("MOVIE ROI CALCULATOR")
        print("-" * 40)
        print(f"Budget: ${budget_input:,.0f}")
        print(f"Runtime: {runtime_input} minutes")
        print(f"Release Quarter: Q{release_quarter_input}")
        print(f"Original Language: {language_input}")
        print(f"Director Count: {director_count_input}")
        print(f"Genres: {', '.join(selected_genres)}")
        print("-" * 40)
        print(f"Predicted ROI: {pred_roi:.2f} ({pred_roi * 100:.1f}%)")
        print(f"Estimated Revenue: ${pred_revenue:,.0f}")
        print(f"Estimated Profit: ${pred_profit:,.0f}")
        print(f"Revenue Range: ${lower_revenue:,.0f} - ${upper_revenue:,.0f}")

        if pred_roi > 0:
            print("Prediction: PROFITABLE")
        else:
            print("Prediction: NOT PROFITABLE")

predict_button.on_click(predict_movie)

display(
    budget_widget,
    runtime_widget,
    quarter_widget,
    language_widget,
    director_widget,
    genre_widget,
    predict_button,
    output
)

We need to try and make this into a streamlit app, might need to migrate to VScode; it has been difficult to get a streamlit app working in Colab